# Climate Crossroads: data preprocessing

Builds `climate_paths.csv` from CMIP6 (Google Cloud public bucket).

- Variable: `tas` (near-surface air temperature), monthly (`Amon`)
- Scenarios: historical + SSP1-2.6, SSP2-4.5, SSP3-7.0, SSP5-8.5
- 7 models, member `r1i1p1f1`
- Steps: annual means, area-weighted (cos lat) averages over IPCC AR6 land regions, years capped at 2100, anomaly vs each model's 1850-1900 mean, median/min/max across models, 10-year centered rolling mean

In [ ]:
!pip install gcsfs zarr regionmask cftime

In [ ]:
import pandas as pd, xarray as xr, gcsfs, regionmask, numpy as np

df = pd.read_csv("https://storage.googleapis.com/cmip6/pangeo-cmip6.csv")
fs = gcsfs.GCSFileSystem(token="anon")

## Find models that have all five scenarios

In [ ]:
scenarios = ["historical", "ssp126", "ssp245", "ssp370", "ssp585"]
base = df.query("table_id == 'Amon' and variable_id == 'tas' and experiment_id in @scenarios")

counts = base.groupby(["source_id", "member_id"])["experiment_id"].nunique().reset_index()
complete = counts[counts.experiment_id == len(scenarios)]
print(complete.source_id.nunique(), "models have all 5 scenarios")

In [ ]:
preferred = ["GFDL-ESM4", "MPI-ESM1-2-LR", "IPSL-CM6A-LR", "MIROC6",
             "CanESM5", "MRI-ESM2-0", "ACCESS-CM2"]
avail = complete[complete.source_id.isin(preferred)].copy()
avail["is_r1"] = avail.member_id != "r1i1p1f1"   # prefer r1i1p1f1
avail = avail.sort_values(["is_r1", "member_id"]).drop_duplicates("source_id")
picks = dict(zip(avail.source_id, avail.member_id))
print(picks)

## Regional annual means for each model and scenario (slow: ~10+ min)

In [ ]:
regions = regionmask.defined_regions.ar6.land
rows = []

for model, member in picks.items():
    for scen in scenarios:
        sub = base.query("source_id == @model and member_id == @member and experiment_id == @scen")
        if sub.empty:
            print("missing", model, scen)
            continue
        print("processing", model, scen)
        try:
            ds = xr.open_zarr(fs.get_mapper(sub.zstore.iloc[0]), consolidated=True)
            annual = (ds.tas - 273.15).groupby("time.year").mean().load()

            mask = regions.mask_3D(ds)
            w = np.cos(np.deg2rad(ds.lat))
            reg = annual.weighted(mask * w).mean(dim=("lat", "lon"))

            out = reg.to_dataframe(name="tas").reset_index()[["year", "names", "tas"]]
            out["model"], out["scenario"] = model, scen
            rows.append(out)
        except Exception as e:
            print("  FAILED:", type(e).__name__, e)

## Anomalies, cross-model summary, smoothing, export

In [ ]:
data = pd.concat(rows).rename(columns={"names": "region"})
data = data[data.year <= 2100]   # a few models run past 2100; drop so all scenarios use the same models
print("models in data:", data.model.unique())

baseline = (data.query("scenario == 'historical' and 1850 <= year <= 1900")
                .groupby(["model", "region"]).tas.mean().rename("base"))
data = data.join(baseline, on=["model", "region"])
data["anomaly"] = data.tas - data.base

summary = (data.groupby(["region", "scenario", "year"]).anomaly
               .agg(median="median", low="min", high="max").reset_index())
summary[["median", "low", "high"]] = (summary.groupby(["region", "scenario"])[["median", "low", "high"]]
                                          .transform(lambda s: s.rolling(10, center=True, min_periods=1).mean()))
summary.round(3).to_csv("climate_paths.csv", index=False)

## Sanity check

In [ ]:
print(summary.region.nunique(), "regions, years", summary.year.min(), "to", summary.year.max())
print(summary.query("year == 2100").groupby("scenario")["median"].mean())